# Machine learning in the browser

This notebook installs [scikit-learn](https://scikit-learn.org/) at runtime and
trains a classifier — entirely in your browser, no server required.

scikit-learn is a C extension package (compiled via Cython). Installing it
demonstrates that conda-express can load complex native libraries into
WebAssembly at runtime, not just simple ones like zlib.

*Examples adapted from the
[scikit-learn documentation](https://scikit-learn.org/stable/auto_examples/)
(BSD 3-Clause license).*

## Install scikit-learn

This pulls in scikit-learn and its dependencies (joblib, scipy, etc.) from
[emscripten-forge](https://emscripten-forge.org/). The shared libraries are
automatically loaded into the WASM runtime.

> **Note:** scikit-learn is a large package with many compiled extensions.
> If installation fails due to a missing library, the emscripten-forge build
> may need updating — check [emscripten-forge.org](https://emscripten-forge.org/)
> for the latest status.

In [ ]:
%load_ext conda_emscripten

In [ ]:
%conda install scikit-learn

## Classifier comparison

Train three different classifiers on the same synthetic dataset and compare
their decision boundaries. This is a classic way to visualize how algorithms
differ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

datasets = [
    ("Moons", make_moons(n_samples=200, noise=0.25, random_state=0)),
    ("Circles", make_circles(n_samples=200, noise=0.2, factor=0.5, random_state=1)),
    ("Linear", make_classification(
        n_samples=200, n_features=2, n_redundant=0, n_informative=2,
        random_state=2, n_clusters_per_class=1,
    )),
]

classifiers = [
    ("k-NN (k=5)", KNeighborsClassifier(5)),
    ("RBF SVM", SVC(gamma="auto")),
    ("Random Forest", RandomForestClassifier(n_estimators=50, random_state=42)),
]

fig, axes = plt.subplots(len(datasets), len(classifiers) + 1,
                         figsize=(14, 3 * len(datasets)))
cm_data = ListedColormap(["#3b82f6", "#ef4444"])
cm_bg = ListedColormap(["#dbeafe", "#fee2e2"])
h = 0.02

for row, (ds_name, (X, y)) in enumerate(datasets):
    X = StandardScaler().fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    ax = axes[row, 0]
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_data, s=20, edgecolors="k", linewidths=0.5)
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_data, s=20, alpha=0.4, edgecolors="k", linewidths=0.5)
    ax.set_title(ds_name, fontsize=11)
    ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)

    for col, (clf_name, clf) in enumerate(classifiers, start=1):
        clf.fit(X_train, y_train)
        score = clf.score(X_test, y_test)
        Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        ax = axes[row, col]
        ax.contourf(xx, yy, Z, cmap=cm_bg, alpha=0.8)
        ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_data, s=20, edgecolors="k", linewidths=0.5)
        ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_data, s=20, alpha=0.4, edgecolors="k", linewidths=0.5)
        ax.set_title(f"{clf_name}\n{score:.0%}", fontsize=10)
        ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Classifier comparison — scikit-learn in WebAssembly", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Cross-validation

Evaluate a Random Forest with 5-fold cross-validation — the standard way to
assess model performance.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

iris = load_iris()
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", gamma="auto"),
    "k-NN (k=5)": KNeighborsClassifier(5),
}

print(f"Dataset: Iris ({iris.data.shape[0]} samples, {iris.data.shape[1]} features)")
print(f"{'Model':<22s} {'Mean accuracy':>14s}  {'Std':>6s}")
print("-" * 46)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, iris.data, iris.target, cv=5, scoring="accuracy")
    results[name] = scores
    print(f"  {name:<20s} {scores.mean():>13.1%}  ±{scores.std():>5.1%}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(results.values(), labels=results.keys(), patch_artist=True,
           boxprops=dict(facecolor="#dbeafe", edgecolor="#3b82f6"),
           medianprops=dict(color="#ef4444", linewidth=2))
ax.set_ylabel("Accuracy")
ax.set_title("5-fold cross-validation on Iris dataset")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Clustering with k-means

Unsupervised learning: discover structure in unlabeled data.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

X_blobs, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.scatter(X_blobs[:, 0], X_blobs[:, 1], s=20, color="gray", alpha=0.6)
ax1.set_title("Unlabeled data")

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_blobs)
colors = ["#3b82f6", "#ef4444", "#22c55e", "#f59e0b"]

for k in range(4):
    mask = labels == k
    ax2.scatter(X_blobs[mask, 0], X_blobs[mask, 1], s=20, color=colors[k], alpha=0.7)
ax2.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            s=200, c="black", marker="X", edgecolors="white", linewidths=2, zorder=5)
ax2.set_title(f"k-means (k=4, inertia={kmeans.inertia_:.0f})")

for ax in (ax1, ax2):
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Unsupervised clustering — scikit-learn in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## What just happened

You installed a 15+ MB package with compiled Cython extensions, trained
classifiers, ran cross-validation, and performed clustering — all in your
browser tab. The same code would work identically on your laptop with
`conda install scikit-learn`.

Browse available packages at [emscripten-forge.org](https://emscripten-forge.org/)
or use `%conda search <package>` to explore from this notebook.